In [0]:
# Create the gold schema if it doesn't exist
spark.sql("""
CREATE SCHEMA IF NOT EXISTS cricketscoreprediction.gold
COMMENT 'Gold layer: Business-ready aggregated tables for analytics and ML'
""")
print("✓ Gold schema created")

In [0]:
# Create comprehensive match summary table
spark.sql("""
CREATE OR REPLACE TABLE cricketscoreprediction.gold.match_summary AS
SELECT 
  mi.match_key,
  mi.match_date,
  YEAR(mi.match_date) as match_year,
  MONTH(mi.match_date) as match_month,
  DAYOFWEEK(mi.match_date) as match_day_of_week,
  mi.match_type,
  mi.venue,
  mi.city,
  mi.team1,
  mi.team2,
  mi.toss_winner,
  mi.toss_decision,
  mi.outcome_winner,
  mi.outcome_result,
  mi.outcome_by_runs,
  mi.outcome_by_wickets,
  mi.overs as total_overs,
  mi.player_of_match,
  mi.umpire1,
  mi.umpire2,
  -- Calculate toss impact
  CASE WHEN mi.toss_winner = mi.outcome_winner THEN 1 ELSE 0 END as toss_won_match,
  -- Identify winning margin type
  CASE 
    WHEN mi.outcome_by_runs IS NOT NULL THEN 'runs'
    WHEN mi.outcome_by_wickets IS NOT NULL THEN 'wickets'
    ELSE 'other'
  END as win_type,
  COALESCE(mi.outcome_by_runs, 0) as winning_runs,
  COALESCE(mi.outcome_by_wickets, 0) as winning_wickets,
  current_timestamp() as created_at
FROM cricketscoreprediction.silver.matches_info mi
WHERE mi.is_current = true
""")

count = spark.sql("SELECT COUNT(*) as cnt FROM cricketscoreprediction.gold.match_summary").collect()[0]['cnt']
print(f"✓ Match summary table created with {count} records")

In [0]:
# Create comprehensive player batting statistics
spark.sql("""
CREATE OR REPLACE TABLE cricketscoreprediction.gold.player_batting_stats AS
WITH batting_deliveries AS (
  SELECT 
    innings.batter as player_name,
    innings.team,
    innings.over_number,
    innings.runs_batter,
    innings.runs_total,
    CASE WHEN innings.runs_batter = 4 THEN 1 ELSE 0 END as is_four,
    CASE WHEN innings.runs_batter = 6 THEN 1 ELSE 0 END as is_six,
    CASE WHEN innings.runs_batter = 0 AND innings.runs_total = 0 THEN 1 ELSE 0 END as is_dot,
    CASE WHEN SIZE(innings.wickets) > 0 THEN 1 ELSE 0 END as is_out,
    CASE WHEN innings.over_number < 6 THEN 1 ELSE 0 END as is_powerplay,
    CASE 
      WHEN innings.over_number >= 6 AND innings.over_number < 16 THEN 1 
      ELSE 0 
    END as is_middle_overs,
    CASE WHEN innings.over_number >= 16 THEN 1 ELSE 0 END as is_death_overs
  FROM cricketscoreprediction.silver.matches_innings innings
  WHERE innings.is_current = true
    AND innings.batter IS NOT NULL
)
SELECT 
  player_name,
  -- Overall statistics
  COUNT(DISTINCT team) as teams_played_for,
  COUNT(*) as balls_faced,
  SUM(runs_batter) as total_runs,
  ROUND(SUM(runs_batter) * 1.0 / NULLIF(COUNT(*), 0), 2) as batting_average,
  ROUND(SUM(runs_batter) * 100.0 / NULLIF(COUNT(*), 0), 2) as strike_rate,
  SUM(is_four) as fours,
  SUM(is_six) as sixes,
  SUM(is_dot) as dots,
  SUM(is_out) as times_dismissed,
  ROUND(COUNT(*) * 1.0 / NULLIF(SUM(is_out), 0), 2) as balls_per_dismissal,
  -- Boundary percentage
  ROUND((SUM(is_four) + SUM(is_six)) * 100.0 / NULLIF(COUNT(*), 0), 2) as boundary_percentage,
  ROUND((SUM(is_four) * 4 + SUM(is_six) * 6) * 100.0 / NULLIF(SUM(runs_batter), 0), 2) as runs_from_boundaries_pct,
  -- Phase-wise statistics
  SUM(CASE WHEN is_powerplay = 1 THEN runs_batter ELSE 0 END) as powerplay_runs,
  SUM(CASE WHEN is_powerplay = 1 THEN 1 ELSE 0 END) as powerplay_balls,
  ROUND(SUM(CASE WHEN is_powerplay = 1 THEN runs_batter ELSE 0 END) * 100.0 / 
        NULLIF(SUM(CASE WHEN is_powerplay = 1 THEN 1 ELSE 0 END), 0), 2) as powerplay_sr,
  SUM(CASE WHEN is_middle_overs = 1 THEN runs_batter ELSE 0 END) as middle_overs_runs,
  SUM(CASE WHEN is_middle_overs = 1 THEN 1 ELSE 0 END) as middle_overs_balls,
  ROUND(SUM(CASE WHEN is_middle_overs = 1 THEN runs_batter ELSE 0 END) * 100.0 / 
        NULLIF(SUM(CASE WHEN is_middle_overs = 1 THEN 1 ELSE 0 END), 0), 2) as middle_overs_sr,
  SUM(CASE WHEN is_death_overs = 1 THEN runs_batter ELSE 0 END) as death_overs_runs,
  SUM(CASE WHEN is_death_overs = 1 THEN 1 ELSE 0 END) as death_overs_balls,
  ROUND(SUM(CASE WHEN is_death_overs = 1 THEN runs_batter ELSE 0 END) * 100.0 / 
        NULLIF(SUM(CASE WHEN is_death_overs = 1 THEN 1 ELSE 0 END), 0), 2) as death_overs_sr,
  -- Consistency metrics
  ROUND(STDDEV(runs_batter), 2) as run_variance,
  current_timestamp() as created_at
FROM batting_deliveries
GROUP BY player_name
HAVING COUNT(*) >= 10  -- Filter players with at least 10 balls faced
""")

count = spark.sql("SELECT COUNT(*) as cnt FROM cricketscoreprediction.gold.player_batting_stats").collect()[0]['cnt']
print(f"✓ Player batting statistics created with {count} players")

In [0]:
# Create comprehensive player bowling statistics
spark.sql("""
CREATE OR REPLACE TABLE cricketscoreprediction.gold.player_bowling_stats AS
WITH bowling_deliveries AS (
  SELECT 
    innings.bowler as player_name,
    innings.team,
    innings.over_number,
    innings.runs_total,
    innings.extras_wides,
    innings.extras_noballs,
    CASE WHEN SIZE(innings.wickets) > 0 THEN 1 ELSE 0 END as is_wicket,
    CASE WHEN innings.runs_total = 0 THEN 1 ELSE 0 END as is_dot,
    CASE WHEN innings.runs_batter = 4 THEN 1 ELSE 0 END as is_four,
    CASE WHEN innings.runs_batter = 6 THEN 1 ELSE 0 END as is_six,
    CASE WHEN innings.over_number < 6 THEN 1 ELSE 0 END as is_powerplay,
    CASE 
      WHEN innings.over_number >= 6 AND innings.over_number < 16 THEN 1 
      ELSE 0 
    END as is_middle_overs,
    CASE WHEN innings.over_number >= 16 THEN 1 ELSE 0 END as is_death_overs,
    -- Wicket type analysis
    CASE 
      WHEN SIZE(innings.wickets) > 0 THEN innings.wickets[0].kind
      ELSE NULL
    END as wicket_type
  FROM cricketscoreprediction.silver.matches_innings innings
  WHERE innings.is_current = true
    AND innings.bowler IS NOT NULL
)
SELECT 
  player_name,
  -- Overall statistics
  COUNT(*) as balls_bowled,
  ROUND(COUNT(*) / 6.0, 2) as overs_bowled,
  SUM(runs_total) as runs_conceded,
  SUM(is_wicket) as wickets_taken,
  ROUND(SUM(runs_total) * 1.0 / NULLIF(SUM(is_wicket), 0), 2) as bowling_average,
  ROUND(SUM(runs_total) * 6.0 / NULLIF(COUNT(*), 0), 2) as economy_rate,
  ROUND(COUNT(*) * 1.0 / NULLIF(SUM(is_wicket), 0), 2) as strike_rate,
  SUM(is_dot) as dots_bowled,
  ROUND(SUM(is_dot) * 100.0 / NULLIF(COUNT(*), 0), 2) as dot_ball_percentage,
  -- Extras
  SUM(COALESCE(extras_wides, 0)) as total_wides,
  SUM(COALESCE(extras_noballs, 0)) as total_noballs,
  ROUND((SUM(COALESCE(extras_wides, 0)) + SUM(COALESCE(extras_noballs, 0))) * 100.0 / 
        NULLIF(COUNT(*), 0), 2) as extras_percentage,
  -- Boundary conceded
  SUM(is_four) as fours_conceded,
  SUM(is_six) as sixes_conceded,
  ROUND((SUM(is_four) + SUM(is_six)) * 100.0 / NULLIF(COUNT(*), 0), 2) as boundary_conceded_pct,
  -- Phase-wise statistics
  SUM(CASE WHEN is_powerplay = 1 THEN runs_total ELSE 0 END) as powerplay_runs_conceded,
  SUM(CASE WHEN is_powerplay = 1 THEN 1 ELSE 0 END) as powerplay_balls,
  ROUND(SUM(CASE WHEN is_powerplay = 1 THEN runs_total ELSE 0 END) * 6.0 / 
        NULLIF(SUM(CASE WHEN is_powerplay = 1 THEN 1 ELSE 0 END), 0), 2) as powerplay_economy,
  SUM(CASE WHEN is_middle_overs = 1 THEN runs_total ELSE 0 END) as middle_overs_runs_conceded,
  SUM(CASE WHEN is_middle_overs = 1 THEN 1 ELSE 0 END) as middle_overs_balls,
  ROUND(SUM(CASE WHEN is_middle_overs = 1 THEN runs_total ELSE 0 END) * 6.0 / 
        NULLIF(SUM(CASE WHEN is_middle_overs = 1 THEN 1 ELSE 0 END), 0), 2) as middle_overs_economy,
  SUM(CASE WHEN is_death_overs = 1 THEN runs_total ELSE 0 END) as death_overs_runs_conceded,
  SUM(CASE WHEN is_death_overs = 1 THEN 1 ELSE 0 END) as death_overs_balls,
  ROUND(SUM(CASE WHEN is_death_overs = 1 THEN runs_total ELSE 0 END) * 6.0 / 
        NULLIF(SUM(CASE WHEN is_death_overs = 1 THEN 1 ELSE 0 END), 0), 2) as death_overs_economy,
  current_timestamp() as created_at
FROM bowling_deliveries
GROUP BY player_name
HAVING COUNT(*) >= 12  -- Filter bowlers with at least 2 overs
""")

count = spark.sql("SELECT COUNT(*) as cnt FROM cricketscoreprediction.gold.player_bowling_stats").collect()[0]['cnt']
print(f"✓ Player bowling statistics created with {count} bowlers")

In [0]:
# Create comprehensive team performance metrics
spark.sql("""
CREATE OR REPLACE TABLE cricketscoreprediction.gold.team_performance AS
WITH team_matches AS (
  SELECT 
    team1 as team,
    match_key,
    match_date,
    venue,
    city,
    team2 as opponent,
    toss_winner,
    toss_decision,
    outcome_winner,
    CASE WHEN team1 = outcome_winner THEN 1 ELSE 0 END as won,
    CASE WHEN team1 = toss_winner THEN 1 ELSE 0 END as won_toss,
    outcome_by_runs,
    outcome_by_wickets
  FROM cricketscoreprediction.gold.match_summary
  UNION ALL
  SELECT 
    team2 as team,
    match_key,
    match_date,
    venue,
    city,
    team1 as opponent,
    toss_winner,
    toss_decision,
    outcome_winner,
    CASE WHEN team2 = outcome_winner THEN 1 ELSE 0 END as won,
    CASE WHEN team2 = toss_winner THEN 1 ELSE 0 END as won_toss,
    outcome_by_runs,
    outcome_by_wickets
  FROM cricketscoreprediction.gold.match_summary
)
SELECT 
  team,
  COUNT(*) as total_matches,
  SUM(won) as matches_won,
  COUNT(*) - SUM(won) as matches_lost,
  ROUND(SUM(won) * 100.0 / NULLIF(COUNT(*), 0), 2) as win_percentage,
  -- Toss statistics
  SUM(won_toss) as tosses_won,
  ROUND(SUM(won_toss) * 100.0 / NULLIF(COUNT(*), 0), 2) as toss_win_percentage,
  SUM(CASE WHEN won_toss = 1 AND won = 1 THEN 1 ELSE 0 END) as wins_after_winning_toss,
  ROUND(SUM(CASE WHEN won_toss = 1 AND won = 1 THEN 1 ELSE 0 END) * 100.0 / 
        NULLIF(SUM(won_toss), 0), 2) as win_rate_after_winning_toss,
  -- Win margin analysis
  ROUND(AVG(CASE WHEN won = 1 AND outcome_by_runs IS NOT NULL THEN outcome_by_runs END), 2) as avg_win_margin_runs,
  ROUND(AVG(CASE WHEN won = 1 AND outcome_by_wickets IS NOT NULL THEN outcome_by_wickets END), 2) as avg_win_margin_wickets,
  MAX(outcome_by_runs) as biggest_win_by_runs,
  MAX(outcome_by_wickets) as biggest_win_by_wickets,
  -- Home vs Away (considering most frequent venue as home)
  current_timestamp() as created_at
FROM team_matches
GROUP BY team
""")

count = spark.sql("SELECT COUNT(*) as cnt FROM cricketscoreprediction.gold.team_performance").collect()[0]['cnt']
print(f"✓ Team performance metrics created for {count} teams")

In [0]:
# Create venue-specific statistics
spark.sql("""
CREATE OR REPLACE TABLE cricketscoreprediction.gold.venue_stats AS
WITH match_scores AS (
  SELECT 
    mi.match_key,
    mi.venue,
    mi.city,
    mi.match_date,
    mi.toss_decision,
    mi.outcome_winner,
    mi.outcome_by_runs,
    mi.outcome_by_wickets,
    mi.team1,
    mi.team2
  FROM cricketscoreprediction.gold.match_summary mi
),
team_innings_scores AS (
  SELECT 
    team,
    SUM(runs_total) as total_score,
    COUNT(*) as balls_faced,
    SUM(CASE WHEN SIZE(wickets) > 0 THEN 1 ELSE 0 END) as wickets_lost
  FROM cricketscoreprediction.silver.matches_innings
  WHERE is_current = true
  GROUP BY team
),
venue_innings AS (
  SELECT 
    ms.venue,
    ms.city,
    ms.match_key,
    ms.match_date,
    ms.toss_decision,
    ms.outcome_winner,
    ms.outcome_by_runs,
    ms.outcome_by_wickets,
    t1.total_score as team1_score,
    t1.wickets_lost as team1_wickets,
    t2.total_score as team2_score,
    t2.wickets_lost as team2_wickets
  FROM match_scores ms
  LEFT JOIN team_innings_scores t1 ON ms.team1 = t1.team
  LEFT JOIN team_innings_scores t2 ON ms.team2 = t2.team
)
SELECT 
  venue,
  city,
  COUNT(DISTINCT match_key) as total_matches,
  -- Scoring statistics (average across both innings)
  ROUND(AVG((COALESCE(team1_score, 0) + COALESCE(team2_score, 0)) / 2.0), 2) as avg_innings_score,
  GREATEST(MAX(team1_score), MAX(team2_score)) as highest_score,
  LEAST(COALESCE(MIN(team1_score), 999999), COALESCE(MIN(team2_score), 999999)) as lowest_score,
  ROUND(STDDEV((COALESCE(team1_score, 0) + COALESCE(team2_score, 0)) / 2.0), 2) as score_variance,
  ROUND(AVG((COALESCE(team1_wickets, 0) + COALESCE(team2_wickets, 0)) / 2.0), 2) as avg_wickets_lost,
  -- Toss impact at venue
  SUM(CASE WHEN toss_decision = 'bat' THEN 1 ELSE 0 END) as times_bat_first_chosen,
  SUM(CASE WHEN toss_decision = 'field' THEN 1 ELSE 0 END) as times_field_first_chosen,
  ROUND(SUM(CASE WHEN toss_decision = 'bat' THEN 1 ELSE 0 END) * 100.0 / 
        NULLIF(COUNT(DISTINCT match_key), 0), 2) as bat_first_percentage,
  -- Win margin at venue
  ROUND(AVG(outcome_by_runs), 2) as avg_win_margin_runs,
  ROUND(AVG(outcome_by_wickets), 2) as avg_win_margin_wickets,
  current_timestamp() as created_at
FROM venue_innings
GROUP BY venue, city
HAVING COUNT(DISTINCT match_key) >= 5  -- Filter venues with at least 5 matches
""")

count = spark.sql("SELECT COUNT(*) as cnt FROM cricketscoreprediction.gold.venue_stats").collect()[0]['cnt']
print(f"✓ Venue statistics created for {count} venues")

In [0]:
# Create head-to-head records between teams
spark.sql("""
CREATE OR REPLACE TABLE cricketscoreprediction.gold.head_to_head AS
WITH team_pairs AS (
  SELECT 
    CASE WHEN team1 < team2 THEN team1 ELSE team2 END as team_a,
    CASE WHEN team1 < team2 THEN team2 ELSE team1 END as team_b,
    match_key,
    match_date,
    venue,
    outcome_winner,
    outcome_by_runs,
    outcome_by_wickets,
    toss_winner
  FROM cricketscoreprediction.gold.match_summary
)
SELECT 
  team_a,
  team_b,
  COUNT(*) as total_matches,
  SUM(CASE WHEN outcome_winner = team_a THEN 1 ELSE 0 END) as team_a_wins,
  SUM(CASE WHEN outcome_winner = team_b THEN 1 ELSE 0 END) as team_b_wins,
  ROUND(SUM(CASE WHEN outcome_winner = team_a THEN 1 ELSE 0 END) * 100.0 / 
        NULLIF(COUNT(*), 0), 2) as team_a_win_percentage,
  ROUND(SUM(CASE WHEN outcome_winner = team_b THEN 1 ELSE 0 END) * 100.0 / 
        NULLIF(COUNT(*), 0), 2) as team_b_win_percentage,
  -- Win margins
  ROUND(AVG(CASE WHEN outcome_winner = team_a AND outcome_by_runs IS NOT NULL 
                  THEN outcome_by_runs END), 2) as team_a_avg_win_margin_runs,
  ROUND(AVG(CASE WHEN outcome_winner = team_b AND outcome_by_runs IS NOT NULL 
                  THEN outcome_by_runs END), 2) as team_b_avg_win_margin_runs,
  -- Recent form (last 5 matches)
  MAX(match_date) as last_match_date,
  current_timestamp() as created_at
FROM team_pairs
GROUP BY team_a, team_b
HAVING COUNT(*) >= 3  -- Filter pairs with at least 3 matches
""")

count = spark.sql("SELECT COUNT(*) as cnt FROM cricketscoreprediction.gold.head_to_head").collect()[0]['cnt']
print(f"✓ Head-to-head records created for {count} team pairs")

In [0]:
# Create innings progression KPI for score prediction
spark.sql("""
CREATE OR REPLACE TABLE cricketscoreprediction.gold.innings_progression_kpi AS
WITH ball_by_ball AS (
  SELECT 
    mi.venue,
    mi.match_date,
    innings.team,
    innings.over_number,
    innings.runs_total,
    innings.runs_batter,
    CASE WHEN SIZE(innings.wickets) > 0 THEN 1 ELSE 0 END as is_wicket,
    ROW_NUMBER() OVER (PARTITION BY mi.venue, mi.match_date, innings.team ORDER BY innings.over_number, innings.actual_delivery) as ball_number
  FROM cricketscoreprediction.gold.match_summary mi
  CROSS JOIN cricketscoreprediction.silver.matches_innings innings
  WHERE innings.is_current = true
    AND (innings.team = mi.team1 OR innings.team = mi.team2)
),
running_totals AS (
  SELECT 
    venue,
    match_date,
    team,
    over_number,
    ball_number,
    SUM(runs_total) OVER (PARTITION BY venue, match_date, team ORDER BY ball_number) as cumulative_runs,
    SUM(is_wicket) OVER (PARTITION BY venue, match_date, team ORDER BY ball_number) as cumulative_wickets,
    ROUND(ball_number / 6.0, 2) as overs_completed
  FROM ball_by_ball
)
SELECT 
  venue,
  over_number,
  -- Aggregate statistics at each over
  ROUND(AVG(cumulative_runs), 2) as avg_score_at_over,
  ROUND(MIN(cumulative_runs), 2) as min_score_at_over,
  ROUND(MAX(cumulative_runs), 2) as max_score_at_over,
  ROUND(STDDEV(cumulative_runs), 2) as score_stddev_at_over,
  ROUND(AVG(cumulative_wickets), 2) as avg_wickets_at_over,
  ROUND(PERCENTILE(cumulative_runs, 0.25), 2) as score_25th_percentile,
  ROUND(PERCENTILE(cumulative_runs, 0.50), 2) as score_median,
  ROUND(PERCENTILE(cumulative_runs, 0.75), 2) as score_75th_percentile,
  COUNT(*) as sample_size,
  current_timestamp() as created_at
FROM running_totals
WHERE over_number IS NOT NULL
GROUP BY venue, over_number
""")

count = spark.sql("SELECT COUNT(*) as cnt FROM cricketscoreprediction.gold.innings_progression_kpi").collect()[0]['cnt']
print(f"✓ Innings progression KPI created with {count} records")

In [0]:
# Create powerplay analysis KPI
spark.sql("""
CREATE OR REPLACE TABLE cricketscoreprediction.gold.powerplay_analysis_kpi AS
WITH powerplay_data AS (
  SELECT 
    mi.match_key,
    mi.venue,
    mi.match_date,
    mi.team1,
    mi.team2,
    mi.outcome_winner,
    innings.team,
    CASE WHEN innings.over_number < 6 THEN 'powerplay'
         WHEN innings.over_number >= 6 AND innings.over_number < 16 THEN 'middle'
         ELSE 'death' 
    END as phase,
    innings.runs_total,
    CASE WHEN SIZE(innings.wickets) > 0 THEN 1 ELSE 0 END as is_wicket,
    innings.runs_batter,
    CASE WHEN innings.runs_batter = 4 THEN 1 ELSE 0 END as is_four,
    CASE WHEN innings.runs_batter = 6 THEN 1 ELSE 0 END as is_six
  FROM cricketscoreprediction.gold.match_summary mi
  CROSS JOIN cricketscoreprediction.silver.matches_innings innings
  WHERE innings.is_current = true
    AND (innings.team = mi.team1 OR innings.team = mi.team2)
)
SELECT 
  venue,
  team,
  phase,
  COUNT(*) as balls_faced,
  SUM(runs_total) as total_runs,
  ROUND(AVG(runs_total), 2) as avg_runs_per_ball,
  ROUND(SUM(runs_total) * 6.0 / NULLIF(COUNT(*), 0), 2) as run_rate,
  SUM(is_wicket) as wickets_lost,
  ROUND(SUM(is_wicket) * 6.0 / NULLIF(COUNT(*), 0), 2) as wickets_per_over,
  SUM(is_four) as fours,
  SUM(is_six) as sixes,
  ROUND((SUM(is_four) + SUM(is_six)) * 100.0 / NULLIF(COUNT(*), 0), 2) as boundary_percentage,
  -- Win correlation
  ROUND(AVG(CASE WHEN team = outcome_winner THEN 1.0 ELSE 0.0 END), 2) as win_correlation,
  current_timestamp() as created_at
FROM powerplay_data
GROUP BY venue, team, phase
""")

count = spark.sql("SELECT COUNT(*) as cnt FROM cricketscoreprediction.gold.powerplay_analysis_kpi").collect()[0]['cnt']
print(f"✓ Powerplay analysis KPI created with {count} records")

In [0]:
# Create partnership analysis KPI
spark.sql("""
CREATE OR REPLACE TABLE cricketscoreprediction.gold.partnership_analysis_kpi AS
WITH delivery_sequence AS (
  SELECT 
    mi.venue,
    mi.match_date,
    innings.team,
    innings.batter,
    innings.non_striker,
    innings.over_number,
    innings.runs_batter,
    innings.runs_total,
    CASE WHEN SIZE(innings.wickets) > 0 THEN 1 ELSE 0 END as is_wicket,
    ROW_NUMBER() OVER (PARTITION BY mi.venue, mi.match_date, innings.team 
                       ORDER BY innings.over_number, innings.actual_delivery) as delivery_num
  FROM cricketscoreprediction.gold.match_summary mi
  CROSS JOIN cricketscoreprediction.silver.matches_innings innings
  WHERE innings.is_current = true
    AND (innings.team = mi.team1 OR innings.team = mi.team2)
    AND innings.batter IS NOT NULL 
    AND innings.non_striker IS NOT NULL
),
wicket_positions AS (
  SELECT 
    venue,
    team,
    delivery_num,
    SUM(is_wicket) OVER (PARTITION BY venue, team ORDER BY delivery_num) as wicket_number
  FROM delivery_sequence
),
partnerships AS (
  SELECT 
    ds.venue,
    ds.team,
    wp.wicket_number + 1 as partnership_number,
    CONCAT_WS(' & ', 
      LEAST(ds.batter, ds.non_striker), 
      GREATEST(ds.batter, ds.non_striker)
    ) as batsmen_pair,
    COUNT(*) as balls,
    SUM(ds.runs_total) as partnership_runs,
    MAX(ds.over_number) - MIN(ds.over_number) as overs_lasted
  FROM delivery_sequence ds
  JOIN wicket_positions wp
    ON ds.venue = wp.venue
    AND ds.team = wp.team 
    AND ds.delivery_num = wp.delivery_num
  GROUP BY ds.venue, ds.team, wp.wicket_number, batsmen_pair
)
SELECT 
  venue,
  partnership_number,
  COUNT(*) as total_partnerships,
  ROUND(AVG(partnership_runs), 2) as avg_partnership_runs,
  ROUND(AVG(balls), 2) as avg_partnership_balls,
  ROUND(AVG(partnership_runs) * 6.0 / NULLIF(AVG(balls), 0), 2) as avg_run_rate,
  MAX(partnership_runs) as highest_partnership,
  MIN(partnership_runs) as lowest_partnership,
  ROUND(STDDEV(partnership_runs), 2) as partnership_stddev,
  current_timestamp() as created_at
FROM partnerships
GROUP BY venue, partnership_number
HAVING partnership_number <= 11  -- Limit to 11 partnerships (10 wickets + 1)
""")

count = spark.sql("SELECT COUNT(*) as cnt FROM cricketscoreprediction.gold.partnership_analysis_kpi").collect()[0]['cnt']
print(f"✓ Partnership analysis KPI created with {count} records")

In [0]:
# Create target chase analysis KPI
spark.sql("""
CREATE OR REPLACE TABLE cricketscoreprediction.gold.target_chase_kpi AS
WITH first_innings AS (
  SELECT 
    mi.match_key,
    mi.venue,
    mi.team1,
    mi.team2,
    mi.outcome_winner,
    mi.outcome_by_runs,
    mi.outcome_by_wickets,
    innings.team as batting_team,
    SUM(innings.runs_total) as first_innings_score,
    SUM(CASE WHEN SIZE(innings.wickets) > 0 THEN 1 ELSE 0 END) as first_innings_wickets
  FROM cricketscoreprediction.gold.match_summary mi
  CROSS JOIN cricketscoreprediction.silver.matches_innings innings
  WHERE innings.is_current = true
    AND (innings.team = mi.team1 OR innings.team = mi.team2)
  GROUP BY mi.match_key, mi.venue, mi.team1, mi.team2, mi.outcome_winner, 
           mi.outcome_by_runs, mi.outcome_by_wickets, innings.team
  HAVING COUNT(*) <= 120  -- First innings typically
)
SELECT 
  venue,
  -- Target score ranges
  CASE 
    WHEN first_innings_score < 120 THEN '< 120'
    WHEN first_innings_score >= 120 AND first_innings_score < 150 THEN '120-150'
    WHEN first_innings_score >= 150 AND first_innings_score < 180 THEN '150-180'
    WHEN first_innings_score >= 180 AND first_innings_score < 200 THEN '180-200'
    ELSE '>= 200'
  END as target_range,
  COUNT(*) as total_chases,
  -- Chase success analysis
  SUM(CASE 
    WHEN outcome_winner != batting_team THEN 1 
    ELSE 0 
  END) as successful_chases,
  ROUND(SUM(CASE WHEN outcome_winner != batting_team THEN 1 ELSE 0 END) * 100.0 / 
        NULLIF(COUNT(*), 0), 2) as chase_success_rate,
  -- Average winning/losing margins
  ROUND(AVG(CASE WHEN outcome_by_wickets IS NOT NULL THEN outcome_by_wickets END), 2) as avg_wickets_remaining_when_won,
  ROUND(AVG(CASE WHEN outcome_by_runs IS NOT NULL THEN outcome_by_runs END), 2) as avg_runs_deficit_when_lost,
  -- Score statistics
  ROUND(AVG(first_innings_score), 2) as avg_target,
  MAX(first_innings_score) as highest_target,
  MIN(first_innings_score) as lowest_target,
  current_timestamp() as created_at
FROM first_innings
GROUP BY venue, target_range
""")

count = spark.sql("SELECT COUNT(*) as cnt FROM cricketscoreprediction.gold.target_chase_kpi").collect()[0]['cnt']
print(f"✓ Target chase analysis KPI created with {count} records")

In [0]:
# Create match momentum KPI for predicting match outcomes
spark.sql("""
CREATE OR REPLACE TABLE cricketscoreprediction.gold.match_momentum_kpi AS
WITH over_summary AS (
  SELECT 
    mi.match_key,
    mi.venue,
    mi.outcome_winner,
    innings.team,
    innings.over_number,
    SUM(innings.runs_total) as runs_in_over,
    SUM(CASE WHEN SIZE(innings.wickets) > 0 THEN 1 ELSE 0 END) as wickets_in_over,
    COUNT(*) as balls_in_over
  FROM cricketscoreprediction.gold.match_summary mi
  CROSS JOIN cricketscoreprediction.silver.matches_innings innings
  WHERE innings.is_current = true
    AND (innings.team = mi.team1 OR innings.team = mi.team2)
  GROUP BY mi.match_key, mi.venue, mi.match_date, mi.outcome_winner, innings.team, innings.over_number
),
momentum_windows AS (
  SELECT 
    match_key,
    venue,
    outcome_winner,
    team,
    over_number,
    runs_in_over,
    wickets_in_over,
    -- Calculate moving average of runs (last 3 overs)
    ROUND(AVG(runs_in_over) OVER (
      PARTITION BY venue, match_key, team 
      ORDER BY over_number 
      ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) as runs_momentum_3over,
    -- Calculate cumulative wickets lost
    SUM(wickets_in_over) OVER (
      PARTITION BY venue, match_key, team 
      ORDER BY over_number
    ) as cumulative_wickets,
    -- Calculate run rate acceleration
    ROUND(runs_in_over - LAG(runs_in_over) OVER (
      PARTITION BY venue, match_key, team 
      ORDER BY over_number
    ), 2) as run_rate_change
  FROM over_summary
)
SELECT 
  venue,
  over_number,
  ROUND(AVG(runs_in_over), 2) as avg_runs_per_over,
  ROUND(AVG(runs_momentum_3over), 2) as avg_3over_momentum,
  ROUND(AVG(wickets_in_over), 2) as avg_wickets_per_over,
  ROUND(AVG(cumulative_wickets), 2) as avg_cumulative_wickets,
  ROUND(AVG(run_rate_change), 2) as avg_run_rate_acceleration,
  -- Win correlation
  ROUND(AVG(CASE 
    WHEN team = outcome_winner AND runs_momentum_3over > 8 THEN 1.0 
    ELSE 0.0 
  END), 2) as high_momentum_win_correlation,
  COUNT(*) as sample_size,
  current_timestamp() as created_at
FROM momentum_windows
GROUP BY venue, over_number
""")

count = spark.sql("SELECT COUNT(*) as cnt FROM cricketscoreprediction.gold.match_momentum_kpi").collect()[0]['cnt']
print(f"✓ Match momentum KPI created with {count} records")

In [0]:
# Validate all gold layer tables
print("\n=== Gold Layer Validation ===")
print("\nTables created:")

tables = spark.sql("SHOW TABLES IN cricketscoreprediction.gold").collect()
for table in tables:
    table_name = table['tableName']
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM cricketscoreprediction.gold.{table_name}").collect()[0]['cnt']
    print(f"  ✓ {table_name}: {count:,} records")

print("\n=== Sample from Match Summary ===")
display(spark.sql("""
SELECT match_date, venue, team1, team2, outcome_winner, 
       toss_won_match, winning_runs, winning_wickets
FROM cricketscoreprediction.gold.match_summary 
ORDER BY match_date DESC 
LIMIT 5
"""))

print("\n=== Top Batsmen by Strike Rate ===")
display(spark.sql("""
SELECT player_name, total_runs, balls_faced, batting_average, strike_rate, 
       fours, sixes, boundary_percentage
FROM cricketscoreprediction.gold.player_batting_stats 
WHERE balls_faced >= 100
ORDER BY strike_rate DESC 
LIMIT 10
"""))

print("\n=== Top Bowlers by Economy Rate ===")
display(spark.sql("""
SELECT player_name, overs_bowled, wickets_taken, bowling_average, 
       economy_rate, dot_ball_percentage
FROM cricketscoreprediction.gold.player_bowling_stats 
WHERE overs_bowled >= 10
ORDER BY economy_rate ASC 
LIMIT 10
"""))

print("\n✓ Gold layer validation complete!")